# F1 Podium Prediction - Feature Engineering

**Features to Create:**
1. Driver podium % (year -1, -2, -3)
2. Constructor podium % (year -1, -2, -3)
3. Driver average position (year -1, -2, -3)
4. Constructor average position (year -1, -2, -3)
5. Qualifying position (already exists as grid_position)
6. Qualifying consistency (std dev over 1, 2, 3 years)
7. Weather categorization (dry/wet/variable/unknown)
8. Driver average position by weather (last 3 years)

**Data Strategy:**
- Input: 2006-2023 data for a complete picture of the driver podium trends (use data_cleansing file)
- 2006-2008: Used only for calculating historical features
- Output: 2009-2022 (train), 2023 (test) with full features

**Data Leakage Prevention:**
- All features use data strictly BEFORE the current race
- Year -1 means "previous completed calendar year"
- No current race information in features

In [1]:
# Setup and Imports
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Load processed data from Phase 1
train = pd.read_csv('../data/processed/race_driver_records_train.csv')
test = pd.read_csv('../data/processed/race_driver_records_test.csv')

# Convert date columns
train['date'] = pd.to_datetime(train['date'])
train['dob'] = pd.to_datetime(train['dob'])
test['date'] = pd.to_datetime(test['date'])
test['dob'] = pd.to_datetime(test['dob'])

# Combine for feature engineering
df = pd.concat([train, test], ignore_index=True)

# Sort by date (critical for time-based features)
df = df.sort_values(['driverId', 'date']).reset_index(drop=True)

print(f"Combined dataset: {df.shape}")
print(f"Year range: {df['year'].min()} - {df['year'].max()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

Combined dataset: (7235, 46)
Year range: 2006 - 2023
Date range: 2006-03-12 00:00:00 to 2023-07-30 00:00:00


## Feature 7: Weather Categorization (Easy)

Categorize raw weather text into: Dry, Wet, Variable, Unknown

In [3]:
def categorize_weather(weather_text):
    """Categorize weather into dry/wet/variable/unknown"""
    if pd.isna(weather_text) or weather_text == 'unknown':
        return 'unknown'
    
    text = str(weather_text).lower()
    
    # Wet conditions (rain takes priority)
    if any(word in text for word in ['rain', 'wet', 'shower', 'drizzle']):
        return 'wet'
    
    # Variable conditions
    elif any(word in text for word in ['cloud', 'overcast', 'variable', 'changeable', 'mixed']):
        return 'variable'
    
    # Dry conditions
    elif any(word in text for word in ['dry', 'sunny', 'clear', 'warm', 'hot', 'fine']):
        return 'dry'
    
    else:
        return 'unknown'

# Apply categorization
df['weather_category'] = df['weather_raw'].apply(categorize_weather)

print("Weather categorization complete:")
print(df['weather_category'].value_counts())
print(f"\n✓ Feature 7 created: weather_category")

Weather categorization complete:
weather_category
dry         3961
variable    1990
wet         1090
unknown      194
Name: count, dtype: int64

✓ Feature 7 created: weather_category


## Year-Based Historical Performance

Calculate annual statistics and merge with 1, 2, 3 year offsets:
- Driver podium % (year -1, -2, -3)
- Constructor podium % (year -1, -2, -3)
- Driver avg position (year -1, -2, -3)
- Constructor avg position (year -1, -2, -3)

In [4]:
print("="*60)
print("CALCULATING ANNUAL DRIVER STATISTICS")
print("="*60)

# Calculate annual driver statistics
annual_driver_stats = df.groupby(['driverId', 'year']).agg({
    'is_podium': 'mean',      # Podium percentage
    'position': 'mean',       # Average position (only counting finishers)
    'raceId': 'count'         # Number of races
}).reset_index()

annual_driver_stats.columns = ['driverId', 'year', 'annual_podium_pct', 'annual_avg_pos', 'annual_race_count']

print(f"Annual driver stats calculated: {annual_driver_stats.shape}")
print("\nSample:")
print(annual_driver_stats.head(10))

CALCULATING ANNUAL DRIVER STATISTICS
Annual driver stats calculated: (425, 5)

Sample:
   driverId  year  annual_podium_pct  annual_avg_pos  annual_race_count
0         1  2007           0.705882        3.000000                 17
1         1  2008           0.555556        4.411765                 18
2         1  2009           0.294118        7.642857                 17
3         1  2010           0.473684        3.750000                 19
4         1  2011           0.315789        3.750000                 19
5         1  2012           0.350000        5.333333                 20
6         1  2013           0.263158        5.166667                 19
7         1  2014           0.842105        1.437500                 19
8         1  2015           0.894737        1.722222                 19
9         1  2016           0.809524        2.105263                 21


In [5]:
print("="*60)
print("CALCULATING ANNUAL CONSTRUCTOR STATISTICS")
print("="*60)

# Calculate annual constructor statistics
annual_constructor_stats = df.groupby(['constructorId', 'year']).agg({
    'is_podium': 'mean',
    'position': 'mean',
    'raceId': 'count'
}).reset_index()

annual_constructor_stats.columns = ['constructorId', 'year', 'annual_podium_pct', 'annual_avg_pos', 'annual_race_count']

print(f"Annual constructor stats calculated: {annual_constructor_stats.shape}")
print("\nSample:")
print(annual_constructor_stats.head(10))

CALCULATING ANNUAL CONSTRUCTOR STATISTICS
Annual constructor stats calculated: (193, 5)

Sample:
   constructorId  year  annual_podium_pct  annual_avg_pos  annual_race_count
0              1  2006           0.250000        4.478261                 36
1              1  2007           0.705882        2.968750                 34
2              1  2008           0.361111        5.156250                 36
3              1  2009           0.147059        7.884615                 34
4              1  2010           0.421053        4.121212                 38
5              1  2011           0.473684        3.393939                 38
6              1  2012           0.325000        6.181818                 40
7              1  2013           0.000000        9.631579                 38
8              1  2014           0.052632        8.444444                 38
9              1  2015           0.000000       12.125000                 38


In [6]:
print("="*60)
print("MERGING YEAR-1 FEATURES (DRIVER)")
print("="*60)

# Create year -1 driver features
driver_y1 = annual_driver_stats.copy()
driver_y1['year'] = driver_y1['year'] + 1  # Shift forward (2008 stats become available in 2009)
driver_y1.columns = ['driverId', 'year', 'driver_podium_pct_y1', 'driver_avg_pos_y1', 'driver_races_y1']

# Merge with main dataframe
df = df.merge(driver_y1, on=['driverId', 'year'], how='left')

print(f"After year-1 driver merge: {df.shape}")
print(f"Missing driver_podium_pct_y1: {df['driver_podium_pct_y1'].isna().sum()} ({df['driver_podium_pct_y1'].isna().sum()/len(df)*100:.1f}%)")

MERGING YEAR-1 FEATURES (DRIVER)
After year-1 driver merge: (7235, 50)
Missing driver_podium_pct_y1: 1672 (23.1%)


In [7]:
# Create year -2 driver features
driver_y2 = annual_driver_stats.copy()
driver_y2['year'] = driver_y2['year'] + 2
driver_y2.columns = ['driverId', 'year', 'driver_podium_pct_y2', 'driver_avg_pos_y2', 'driver_races_y2']

df = df.merge(driver_y2, on=['driverId', 'year'], how='left')

print(f"After year-2 driver merge: {df.shape}")
print(f"Missing driver_podium_pct_y2: {df['driver_podium_pct_y2'].isna().sum()} ({df['driver_podium_pct_y2'].isna().sum()/len(df)*100:.1f}%)")

After year-2 driver merge: (7235, 53)
Missing driver_podium_pct_y2: 2830 (39.1%)


In [8]:
# Create year -3 driver features
driver_y3 = annual_driver_stats.copy()
driver_y3['year'] = driver_y3['year'] + 3
driver_y3.columns = ['driverId', 'year', 'driver_podium_pct_y3', 'driver_avg_pos_y3', 'driver_races_y3']

df = df.merge(driver_y3, on=['driverId', 'year'], how='left')

print(f"After year-3 driver merge: {df.shape}")
print(f"Missing driver_podium_pct_y3: {df['driver_podium_pct_y3'].isna().sum()} ({df['driver_podium_pct_y3'].isna().sum()/len(df)*100:.1f}%)")

print("\n✓ Features 1 & 3 created: driver_podium_pct_y1/y2/y3, driver_avg_pos_y1/y2/y3")

After year-3 driver merge: (7235, 56)
Missing driver_podium_pct_y3: 3644 (50.4%)

✓ Features 1 & 3 created: driver_podium_pct_y1/y2/y3, driver_avg_pos_y1/y2/y3


In [9]:
print("="*60)
print("MERGING YEAR-1, -2, -3 FEATURES (CONSTRUCTOR)")
print("="*60)

# Constructor year -1
constructor_y1 = annual_constructor_stats.copy()
constructor_y1['year'] = constructor_y1['year'] + 1
constructor_y1.columns = ['constructorId', 'year', 'constructor_podium_pct_y1', 'constructor_avg_pos_y1', 'constructor_races_y1']
df = df.merge(constructor_y1, on=['constructorId', 'year'], how='left')

# Constructor year -2
constructor_y2 = annual_constructor_stats.copy()
constructor_y2['year'] = constructor_y2['year'] + 2
constructor_y2.columns = ['constructorId', 'year', 'constructor_podium_pct_y2', 'constructor_avg_pos_y2', 'constructor_races_y2']
df = df.merge(constructor_y2, on=['constructorId', 'year'], how='left')

# Constructor year -3
constructor_y3 = annual_constructor_stats.copy()
constructor_y3['year'] = constructor_y3['year'] + 3
constructor_y3.columns = ['constructorId', 'year', 'constructor_podium_pct_y3', 'constructor_avg_pos_y3', 'constructor_races_y3']
df = df.merge(constructor_y3, on=['constructorId', 'year'], how='left')

print(f"After constructor merges: {df.shape}")
print(f"Missing constructor_podium_pct_y1: {df['constructor_podium_pct_y1'].isna().sum()} ({df['constructor_podium_pct_y1'].isna().sum()/len(df)*100:.1f}%)")
print(f"Missing constructor_podium_pct_y2: {df['constructor_podium_pct_y2'].isna().sum()} ({df['constructor_podium_pct_y2'].isna().sum()/len(df)*100:.1f}%)")
print(f"Missing constructor_podium_pct_y3: {df['constructor_podium_pct_y3'].isna().sum()} ({df['constructor_podium_pct_y3'].isna().sum()/len(df)*100:.1f}%)")

print("\n✓ Features 2 & 4 created: constructor_podium_pct_y1/y2/y3, constructor_avg_pos_y1/y2/y3")

MERGING YEAR-1, -2, -3 FEATURES (CONSTRUCTOR)
After constructor merges: (7235, 65)
Missing constructor_podium_pct_y1: 1136 (15.7%)
Missing constructor_podium_pct_y2: 2136 (29.5%)
Missing constructor_podium_pct_y3: 2949 (40.8%)

✓ Features 2 & 4 created: constructor_podium_pct_y1/y2/y3, constructor_avg_pos_y1/y2/y3


## Qualifying Consistency

Standard deviation of grid_position over last 1, 2, 3 years

In [10]:
print("="*60)
print("CALCULATING QUALIFYING CONSISTENCY (STD DEV)")
print("="*60)

# Calculate annual qualifying std dev
annual_qual_consistency = df.groupby(['driverId', 'year']).agg({
    'grid_position': 'std'
}).reset_index()

annual_qual_consistency.columns = ['driverId', 'year', 'annual_qual_std']

# Year -1
qual_y1 = annual_qual_consistency.copy()
qual_y1['year'] = qual_y1['year'] + 1
qual_y1.columns = ['driverId', 'year', 'driver_qual_std_y1']
df = df.merge(qual_y1, on=['driverId', 'year'], how='left')

# Year -2
qual_y2 = annual_qual_consistency.copy()
qual_y2['year'] = qual_y2['year'] + 2
qual_y2.columns = ['driverId', 'year', 'driver_qual_std_y2']
df = df.merge(qual_y2, on=['driverId', 'year'], how='left')

# Year -3
qual_y3 = annual_qual_consistency.copy()
qual_y3['year'] = qual_y3['year'] + 3
qual_y3.columns = ['driverId', 'year', 'driver_qual_std_y3']
df = df.merge(qual_y3, on=['driverId', 'year'], how='left')

print(f"After qualifying consistency merge: {df.shape}")
print(f"Missing driver_qual_std_y1: {df['driver_qual_std_y1'].isna().sum()}")

CALCULATING QUALIFYING CONSISTENCY (STD DEV)
After qualifying consistency merge: (7235, 68)
Missing driver_qual_std_y1: 1779


## Weather-Conditional Performance

Average position in dry/wet races over last 3 years

In [11]:
print("="*60)
print("CALCULATING WEATHER-CONDITIONAL PERFORMANCE")
print("="*60)

# Calculate average position in DRY races by driver and year
dry_races = df[df['weather_category'] == 'dry'].copy()
annual_dry_performance = dry_races.groupby(['driverId', 'year']).agg({
    'position': 'mean',
    'raceId': 'count'
}).reset_index()
annual_dry_performance.columns = ['driverId', 'year', 'annual_avg_pos_dry', 'dry_races_count']

# Calculate average position in WET races by driver and year
wet_races = df[df['weather_category'] == 'wet'].copy()
annual_wet_performance = wet_races.groupby(['driverId', 'year']).agg({
    'position': 'mean',
    'raceId': 'count'
}).reset_index()
annual_wet_performance.columns = ['driverId', 'year', 'annual_avg_pos_wet', 'wet_races_count']

print(f"Dry race performance stats: {annual_dry_performance.shape}")
print(f"Wet race performance stats: {annual_wet_performance.shape}")

CALCULATING WEATHER-CONDITIONAL PERFORMANCE
Dry race performance stats: (417, 4)
Wet race performance stats: (394, 4)


In [12]:
# Merge 3-year average for dry races
# We'll create a rolling 3-year average using years -1, -2, -3

# Create copies with shifted years
dry_y1 = annual_dry_performance[['driverId', 'year', 'annual_avg_pos_dry']].copy()
dry_y1['year'] = dry_y1['year'] + 1

dry_y2 = annual_dry_performance[['driverId', 'year', 'annual_avg_pos_dry']].copy()
dry_y2['year'] = dry_y2['year'] + 2

dry_y3 = annual_dry_performance[['driverId', 'year', 'annual_avg_pos_dry']].copy()
dry_y3['year'] = dry_y3['year'] + 3

# Merge all three
dry_combined = df[['driverId', 'year']].drop_duplicates()
dry_combined = dry_combined.merge(dry_y1, on=['driverId', 'year'], how='left', suffixes=('', '_y1'))
dry_combined = dry_combined.merge(dry_y2, on=['driverId', 'year'], how='left', suffixes=('', '_y2'))
dry_combined = dry_combined.merge(dry_y3, on=['driverId', 'year'], how='left', suffixes=('', '_y3'))

# Calculate 3-year average (mean of year -1, -2, -3)
dry_combined['driver_avg_pos_dry_3y'] = dry_combined[['annual_avg_pos_dry', 'annual_avg_pos_dry_y2', 'annual_avg_pos_dry_y3']].mean(axis=1)

# Merge back to main dataframe
df = df.merge(dry_combined[['driverId', 'year', 'driver_avg_pos_dry_3y']], on=['driverId', 'year'], how='left')

print(f"After dry performance merge: {df.shape}")
print(f"Missing driver_avg_pos_dry_3y: {df['driver_avg_pos_dry_3y'].isna().sum()} ({df['driver_avg_pos_dry_3y'].isna().sum()/len(df)*100:.1f}%)")

After dry performance merge: (7235, 69)
Missing driver_avg_pos_dry_3y: 1456 (20.1%)


In [13]:
# Same for wet races
wet_y1 = annual_wet_performance[['driverId', 'year', 'annual_avg_pos_wet']].copy()
wet_y1['year'] = wet_y1['year'] + 1

wet_y2 = annual_wet_performance[['driverId', 'year', 'annual_avg_pos_wet']].copy()
wet_y2['year'] = wet_y2['year'] + 2

wet_y3 = annual_wet_performance[['driverId', 'year', 'annual_avg_pos_wet']].copy()
wet_y3['year'] = wet_y3['year'] + 3

wet_combined = df[['driverId', 'year']].drop_duplicates()
wet_combined = wet_combined.merge(wet_y1, on=['driverId', 'year'], how='left', suffixes=('', '_y1'))
wet_combined = wet_combined.merge(wet_y2, on=['driverId', 'year'], how='left', suffixes=('', '_y2'))
wet_combined = wet_combined.merge(wet_y3, on=['driverId', 'year'], how='left', suffixes=('', '_y3'))

wet_combined['driver_avg_pos_wet_3y'] = wet_combined[['annual_avg_pos_wet', 'annual_avg_pos_wet_y2', 'annual_avg_pos_wet_y3']].mean(axis=1)

df = df.merge(wet_combined[['driverId', 'year', 'driver_avg_pos_wet_3y']], on=['driverId', 'year'], how='left')

print(f"After wet performance merge: {df.shape}")
print(f"Missing driver_avg_pos_wet_3y: {df['driver_avg_pos_wet_3y'].isna().sum()} ({df['driver_avg_pos_wet_3y'].isna().sum()/len(df)*100:.1f}%)")

print("\n✓ Feature 8 created: driver_avg_pos_dry_3y, driver_avg_pos_wet_3y")

After wet performance merge: (7235, 70)
Missing driver_avg_pos_wet_3y: 1749 (24.2%)

✓ Feature 8 created: driver_avg_pos_dry_3y, driver_avg_pos_wet_3y


## Handle Missing Values

Fill missing historical features with appropriate values

In [14]:
print("="*60)
print("HANDLING MISSING VALUES")
print("="*60)

# List of features to fill
features_to_fill = [
    # Driver features
    'driver_podium_pct_y1', 'driver_podium_pct_y2', 'driver_podium_pct_y3',
    'driver_avg_pos_y1', 'driver_avg_pos_y2', 'driver_avg_pos_y3',
    'driver_races_y1', 'driver_races_y2', 'driver_races_y3',
    # Constructor features
    'constructor_podium_pct_y1', 'constructor_podium_pct_y2', 'constructor_podium_pct_y3',
    'constructor_avg_pos_y1', 'constructor_avg_pos_y2', 'constructor_avg_pos_y3',
    'constructor_races_y1', 'constructor_races_y2', 'constructor_races_y3',
    # Qualifying consistency
    'driver_qual_std_y1', 'driver_qual_std_y2', 'driver_qual_std_y3',
    # Weather performance
    'driver_avg_pos_dry_3y', 'driver_avg_pos_wet_3y'
]

print("Missing values before filling:")
for col in features_to_fill:
    missing = df[col].isna().sum()
    if missing > 0:
        print(f"  {col}: {missing} ({missing/len(df)*100:.1f}%)")

HANDLING MISSING VALUES
Missing values before filling:
  driver_podium_pct_y1: 1672 (23.1%)
  driver_podium_pct_y2: 2830 (39.1%)
  driver_podium_pct_y3: 3644 (50.4%)
  driver_avg_pos_y1: 1693 (23.4%)
  driver_avg_pos_y2: 2850 (39.4%)
  driver_avg_pos_y3: 3665 (50.7%)
  driver_races_y1: 1672 (23.1%)
  driver_races_y2: 2830 (39.1%)
  driver_races_y3: 3644 (50.4%)
  constructor_podium_pct_y1: 1136 (15.7%)
  constructor_podium_pct_y2: 2136 (29.5%)
  constructor_podium_pct_y3: 2949 (40.8%)
  constructor_avg_pos_y1: 1136 (15.7%)
  constructor_avg_pos_y2: 2136 (29.5%)
  constructor_avg_pos_y3: 2949 (40.8%)
  constructor_races_y1: 1136 (15.7%)
  constructor_races_y2: 2136 (29.5%)
  constructor_races_y3: 2949 (40.8%)
  driver_qual_std_y1: 1779 (24.6%)
  driver_qual_std_y2: 2888 (39.9%)
  driver_qual_std_y3: 3665 (50.7%)
  driver_avg_pos_dry_3y: 1456 (20.1%)
  driver_avg_pos_wet_3y: 1749 (24.2%)


In [15]:
# Fill strategy:
# Podium percentages: 0 (rookie or no history)
# Average positions: 15 (mid-field position)
# Race counts: 0 (no races)
# Qualifying std: 5 (moderate consistency)
# Weather performance: Overall avg position (from driver_avg_pos_y1 or 15)

# Podium percentages -> 0
podium_cols = [col for col in features_to_fill if 'podium_pct' in col]
df[podium_cols] = df[podium_cols].fillna(0)

# Race counts -> 0
race_count_cols = [col for col in features_to_fill if 'races_' in col]
df[race_count_cols] = df[race_count_cols].fillna(0)

# Average positions -> 15 (mid-field)
avg_pos_cols = [col for col in features_to_fill if 'avg_pos' in col and 'dry' not in col and 'wet' not in col]
df[avg_pos_cols] = df[avg_pos_cols].fillna(15)

# Qualifying std -> 5 (moderate consistency)
qual_std_cols = [col for col in features_to_fill if 'qual_std' in col]
df[qual_std_cols] = df[qual_std_cols].fillna(5)

# Weather performance -> use overall avg position as fallback
df['driver_avg_pos_dry_3y'] = df['driver_avg_pos_dry_3y'].fillna(df['driver_avg_pos_y1'])
df['driver_avg_pos_dry_3y'] = df['driver_avg_pos_dry_3y'].fillna(15)

df['driver_avg_pos_wet_3y'] = df['driver_avg_pos_wet_3y'].fillna(df['driver_avg_pos_y1'])
df['driver_avg_pos_wet_3y'] = df['driver_avg_pos_wet_3y'].fillna(15)

print("\nMissing values after filling:")
for col in features_to_fill:
    missing = df[col].isna().sum()
    if missing > 0:
        print(f"  {col}: {missing}")

print("\n✓ Missing values handled")


Missing values after filling:

✓ Missing values handled


## Filter to Final Modeling Dataset

Keep only 2009+ data (with full 3-year features) for modeling

In [16]:
print("="*60)
print("FILTERING TO FINAL MODELING DATASET (2009+)")
print("="*60)

print(f"Records before filter: {len(df)} ({df['year'].min()}-{df['year'].max()})")

# Filter to 2009+ (2006-2008 was only for feature calculation)
df_final = df[df['year'] >= 2009].copy()

print(f"Records after filter:  {len(df_final)} ({df_final['year'].min()}-{df_final['year'].max()})")
print(f"\nData dropped: {len(df) - len(df_final)} records from {df[df['year'] < 2009]['year'].unique()}")
print("(2006-2008 data used only for calculating historical features)")

FILTERING TO FINAL MODELING DATASET (2009+)
Records before filter: 7235 (2006-2023)
Records after filter:  6097 (2009-2023)

Data dropped: 1138 records from [2007 2008 2006]
(2006-2008 data used only for calculating historical features)


## Feature Summary & Validation

In [24]:
print("="*60)
print("FEATURE ENGINEERING SUMMARY")
print("="*60)

# List all new features
new_features = [
    'weather_category',
    'driver_podium_pct_y1', 'driver_podium_pct_y2', 'driver_podium_pct_y3',
    'driver_avg_pos_y1', 'driver_avg_pos_y2', 'driver_avg_pos_y3',
    'constructor_podium_pct_y1', 'constructor_podium_pct_y2', 'constructor_podium_pct_y3',
    'constructor_avg_pos_y1', 'constructor_avg_pos_y2', 'constructor_avg_pos_y3',
    'driver_qual_std_y1', 'driver_qual_std_y2', 'driver_qual_std_y3',
    'driver_avg_pos_dry_3y', 'driver_avg_pos_wet_3y'
]

print(f"\nNew features created: {len(new_features)}")
print(f"Total features in dataset: {df_final.shape[1]}")

print("\nNew features:")
for i, feat in enumerate(new_features, 1):
    print(f"  {i}. {feat}")

# Check for any remaining nulls in new features
print("\n" + "="*60)
print("NULL CHECK FOR NEW FEATURES")
print("="*60)
null_check = df_final[new_features].isnull().sum()
if null_check.sum() == 0:
    print("No null values in any new features")
else:
    print("Null values found:")
    print(null_check[null_check > 0])

FEATURE ENGINEERING SUMMARY

New features created: 18
Total features in dataset: 70

New features:
  1. weather_category
  2. driver_podium_pct_y1
  3. driver_podium_pct_y2
  4. driver_podium_pct_y3
  5. driver_avg_pos_y1
  6. driver_avg_pos_y2
  7. driver_avg_pos_y3
  8. constructor_podium_pct_y1
  9. constructor_podium_pct_y2
  10. constructor_podium_pct_y3
  11. constructor_avg_pos_y1
  12. constructor_avg_pos_y2
  13. constructor_avg_pos_y3
  14. driver_qual_std_y1
  15. driver_qual_std_y2
  16. driver_qual_std_y3
  17. driver_avg_pos_dry_3y
  18. driver_avg_pos_wet_3y

NULL CHECK FOR NEW FEATURES
No null values in any new features


In [18]:
# Sample of engineered features
print("\nSample of engineered features:")
feature_sample = df_final[['year', 'driver_name', 'name', 'weather_category', 
                           'driver_podium_pct_y1', 'driver_avg_pos_y1',
                           'constructor_podium_pct_y1', 'driver_qual_std_y1',
                           'driver_avg_pos_dry_3y', 'is_podium']].head(10)
print(feature_sample.to_string())


Sample of engineered features:
    year     driver_name                   name weather_category  driver_podium_pct_y1  driver_avg_pos_y1  constructor_podium_pct_y1  driver_qual_std_y1  driver_avg_pos_dry_3y  is_podium
35  2009  Lewis Hamilton  Australian Grand Prix              dry              0.555556           4.411765                   0.361111            3.262482                  2.975          0
36  2009  Lewis Hamilton   Malaysian Grand Prix              wet              0.555556           4.411765                   0.361111            3.262482                  2.975          0
37  2009  Lewis Hamilton     Chinese Grand Prix              wet              0.555556           4.411765                   0.361111            3.262482                  2.975          0
38  2009  Lewis Hamilton     Bahrain Grand Prix              dry              0.555556           4.411765                   0.361111            3.262482                  2.975          0
39  2009  Lewis Hamilton     Span

## Train/Test Split & Save

In [19]:
# Split into train and test
train_fe = df_final[df_final['year'] < 2023].copy()
test_fe = df_final[df_final['year'] == 2023].copy()

print("="*60)
print("FINAL DATASET SPLIT")
print("="*60)
print(f"\nTraining set: {train_fe.shape} ({train_fe['year'].min()}-{train_fe['year'].max()})")
print(f"  Podium rate: {train_fe['is_podium'].mean()*100:.2f}%")
print(f"  Races: {train_fe['raceId'].nunique()}")
print(f"  Drivers: {train_fe['driverId'].nunique()}")

print(f"\nTest set: {test_fe.shape} (year {test_fe['year'].unique()[0]})")
print(f"  Podium rate: {test_fe['is_podium'].mean()*100:.2f}%")
print(f"  Races: {test_fe['raceId'].nunique()}")
print(f"  Drivers: {test_fe['driverId'].nunique()}")

FINAL DATASET SPLIT

Training set: (5857, 70) (2009-2022)
  Podium rate: 14.14%
  Races: 276
  Drivers: 79

Test set: (240, 70) (year 2023)
  Podium rate: 15.00%
  Races: 12
  Drivers: 21


In [23]:
# Save feature-engineered datasets
train_fe.to_csv('../data/processed/race_driver_records_train_fe.csv', index=False)
test_fe.to_csv('../data/processed/race_driver_records_test_fe.csv', index=False)

print("\n" + "="*60)
print("FILES SAVED")
print("="*60)
print("data/processed/race_driver_records_train_fe.csv")
print("data/processed/race_driver_records_test_fe.csv")
print("\nFeature engineering complete")


FILES SAVED
data/processed/race_driver_records_train_fe.csv
data/processed/race_driver_records_test_fe.csv

Feature engineering complete
